## Homework 5

## 1. Import

In [0]:
import os
import tempfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn

from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, datediff, count
from pyspark.sql import Window

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
)



## 2.Load raw F1 data

In [0]:
BASE = "/Volumes/gr5069/raw/f1_data"

results   = spark.read.csv(f"{BASE}/results.csv",   header=True, inferSchema=True)
drivers   = spark.read.csv(f"{BASE}/drivers.csv",   header=True, inferSchema=True)
races     = spark.read.csv(f"{BASE}/races.csv",     header=True, inferSchema=True)
pit_stops = spark.read.csv(f"{BASE}/pit_stops.csv", header=True, inferSchema=True)
lap_times = spark.read.csv(f"{BASE}/lap_times.csv", header=True, inferSchema=True)

print("results  :", results.count())
print("drivers  :", drivers.count())
print("races    :", races.count())
print("pit_stops:", pit_stops.count())
print("lap_times:", lap_times.count())

## 3. Build the modeling dataset

For each (raceId, driverId) row in `results` I build:

**Target**
- `points_finish`: 1 if `positionOrder` between 1 and 10, else 0

**Features**
- `grid` (starting position)
- `driver_age` at race date
- `n_pit_stops` for that driver in that race
- `avg_pit_ms` for that driver in that race
- `avg_lap_ms` for that driver in that race
- `n_laps` completed by that driver in that race
- race year

In [0]:
pit_agg = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(
        F.count("*").alias("n_pit_stops"),
        F.avg(col("milliseconds").cast("double")).alias("avg_pit_ms"),
    )
)

# Lap time aggregates per (raceId, driverId)
lap_agg = (
    lap_times
    .groupBy("raceId", "driverId")
    .agg(
        F.count("*").alias("n_laps"),
        F.avg(col("milliseconds").cast("double")).alias("avg_lap_ms"),
    )
)

# Race year
races_small = races.select("raceId", "year", col("date").alias("race_date"))

# Driver dob for age calculation
drivers_small = drivers.select("driverId", col("dob").alias("driver_dob"))

df = (
    results.select("raceId", "driverId", "grid", "positionOrder")
    .join(races_small, on="raceId", how="left")
    .join(drivers_small, on="driverId", how="left")
    .join(pit_agg, on=["raceId", "driverId"], how="left")
    .join(lap_agg, on=["raceId", "driverId"], how="left")
    .withColumn("driver_age", (datediff(col("race_date"), col("driver_dob")) / 365.25))
    .withColumn("points_finish", when(col("positionOrder") <= 10, 1).otherwise(0))
    .select(
        "raceId", "driverId", "year",
        col("grid").cast("int"),
        col("driver_age").cast("double"),
        col("n_pit_stops").cast("int"),
        col("avg_pit_ms").cast("double"),
        col("n_laps").cast("int"),
        col("avg_lap_ms").cast("double"),
        "points_finish",
    )
)

df.show(5)
print("rows:", df.count())

In [0]:
# Convert to pandas for sklearn. Drop rows with missing features.
pdf = df.toPandas()
print("before dropna:", pdf.shape)

feature_cols = ["grid", "driver_age", "n_pit_stops", "avg_pit_ms", "n_laps", "avg_lap_ms", "year"]
model_pdf = pdf.dropna(subset=feature_cols + ["points_finish"]).copy()
print("after dropna :", model_pdf.shape)

model_pdf.head()

## 4. Train and Test

In [0]:
X = model_pdf[feature_cols].values
y = model_pdf["points_finish"].values
ids = model_pdf[["raceId", "driverId"]].reset_index(drop=True)

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, ids, test_size=0.25, random_state=42, stratify=y
)

print("train:", X_train.shape, "test:", X_test.shape)
print("positive rate train:", y_train.mean().round(3), " test:", y_test.mean().round(3))

## 5. ML Workflow

In [0]:
USER = "xc2849"
EXPERIMENT_PATH = f"/Users/xc2849@columbia.edu/f1_points_finish"

try:
    mlflow.set_experiment(EXPERIMENT_PATH)
except Exception as e:
    # Fallback: use the default workspace experiment
    print("set_experiment fallback:", e)
    mlflow.set_experiment("/Shared/f1_points_finish")

print("experiment:", mlflow.get_experiment_by_name(EXPERIMENT_PATH))

In [0]:
def log_artifacts_and_metrics(model, model_name, X_te, y_te, feature_names):
    """Compute 4 metrics and 2 artifacts, log them, and return predictions + metrics."""
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    metrics = {
        "accuracy":  accuracy_score(y_te, y_pred),
        "precision": precision_score(y_te, y_pred, zero_division=0),
        "recall":    recall_score(y_te, y_pred, zero_division=0),
        "f1":        f1_score(y_te, y_pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_te, y_proba),
    }
    for k, v in metrics.items():
        mlflow.log_metric(k, float(v))

    tmpdir = tempfile.mkdtemp()

    # Artifact 1: confusion matrix PNG
    cm = confusion_matrix(y_te, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(cm, display_labels=["no points", "points"]).plot(ax=ax, colorbar=False)
    ax.set_title(f"{model_name} confusion matrix")
    cm_path = os.path.join(tmpdir, f"{model_name}_confusion_matrix.png")
    fig.tight_layout(); fig.savefig(cm_path, dpi=120); plt.close(fig)
    mlflow.log_artifact(cm_path)

    # Artifact 2: feature importance / coefficients CSV
    if hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
    elif hasattr(model, "coef_"):
        importance = np.abs(model.coef_).ravel()
    else:
        importance = np.zeros(len(feature_names))
    fi_df = pd.DataFrame({"feature": feature_names, "importance": importance}) \
              .sort_values("importance", ascending=False)
    fi_path = os.path.join(tmpdir, f"{model_name}_feature_importance.csv")
    fi_df.to_csv(fi_path, index=False)
    mlflow.log_artifact(fi_path)

    return y_pred, y_proba, metrics

Model 1 Logistic

In [0]:
lr_params = {
    "C": 1.0,
    "penalty": "l2",
    "solver": "lbfgs",
    "max_iter": 1000,
    "random_state": 42,
}

with mlflow.start_run(run_name="logistic_regression") as run_lr:
    mlflow.log_params(lr_params)
    mlflow.set_tag("model_family", "logistic_regression")
    mlflow.set_tag("author", USER)

    lr = LogisticRegression(**lr_params)
    lr.fit(X_train, y_train)

    mlflow.sklearn.log_model(lr, artifact_path="model")

    lr_pred, lr_proba, lr_metrics = log_artifacts_and_metrics(
        lr, "logistic_regression", X_test, y_test, feature_cols
    )
    lr_run_id = run_lr.info.run_id

print("logistic regression metrics:", {k: round(v, 4) for k, v in lr_metrics.items()})
print("run id:", lr_run_id)

Model 2 Random Forest

In [0]:
rf_params = {
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "random_state": 42,
    "n_jobs": -1,
}

with mlflow.start_run(run_name="random_forest") as run_rf:
    mlflow.log_params(rf_params)
    mlflow.set_tag("model_family", "random_forest")
    mlflow.set_tag("author", USER)

    rf = RandomForestClassifier(**rf_params)
    rf.fit(X_train, y_train)

    mlflow.sklearn.log_model(rf, artifact_path="model")

    rf_pred, rf_proba, rf_metrics = log_artifacts_and_metrics(
        rf, "random_forest", X_test, y_test, feature_cols
    )
    rf_run_id = run_rf.info.run_id

print("random forest metrics:", {k: round(v, 4) for k, v in rf_metrics.items()})
print("run id:", rf_run_id)

## 6.Comparing

In [0]:
compare = pd.DataFrame([lr_metrics, rf_metrics], index=["logistic_regression", "random_forest"])
compare.round(4)

## 7. Create my own database and write predictions to two tables
